# 03 · Bố cục report và preview

Tạo prompt AI 3 từ requirement/model đã duyệt, kiểm tra field dùng trong layout và vẽ wireframe để khách chọn phương án. Preview chỉ thể hiện bố cục, không phải ảnh render của Power BI Desktop.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'README.md').exists() and (candidate / 'src').is_dir():
            return candidate
    raise FileNotFoundError('Không tìm thấy repo root.')

ROOT = find_repo_root()
RUNS_DIR = ROOT / 'runs'
JOB_ID = None  # Có thể đặt job_id cụ thể.
candidates = [p for p in RUNS_DIR.glob('*') if (p / 'model_spec.json').exists()]
if JOB_ID:
    RUN_DIR = RUNS_DIR / JOB_ID
elif candidates:
    RUN_DIR = max(candidates, key=lambda p: p.stat().st_mtime)
else:
    raise FileNotFoundError('Chưa có model_spec.json. Chạy 02_model_design.ipynb trước.')
JOB_ID = RUN_DIR.name
requirement = json.loads((RUN_DIR / 'requirement.json').read_text(encoding='utf-8'))
model = json.loads((RUN_DIR / 'model_spec.json').read_text(encoding='utf-8'))
available_fields = set()
for table in model['tables']:
    available_fields.update(f"{table['name']}[{column['name']}]" for column in table.get('columns', []))
available_fields.update(f"{measure['table']}[{measure['name']}]" for measure in model.get('measures', []))
print('Job:', JOB_ID, '| Có thể dùng:', ', '.join(sorted(available_fields)))

## AI 3 · Đề xuất 2–3 phương án
Field phải dùng cú pháp `Table[Column]` hoặc `Table[Measure]` và có trong model. Vị trí preview dùng tọa độ chuẩn hóa từ 0 đến 1: `x`, `y`, `width`, `height`.

In [ ]:
prompt = f'''Đề xuất 2 hoặc 3 layout Power BI cho requirement/model bên dưới. Ưu tiên dễ đọc, mục tiêu nghiệp vụ rõ, không dùng field ngoài model.
Trả về JSON duy nhất: {{"layouts": [{{"id": "A", "name": "...", "pages": [{{"name": "...", "purpose": "...", "visuals": [{{"type": "card|bar|line|donut|table|slicer", "title": "...", "fields": [{{"field": "Table[Column]", "role": "value|category|legend|filter"}}], "position": {{"x": 0.0, "y": 0.0, "width": 0.3, "height": 0.2}}}}]}}]}}]}}.
Không trả lời markdown. Mỗi phương án có 1–3 trang; vị trí nằm trong khoảng 0..1, tránh chồng lấn.

REQUIREMENT:
{json.dumps(requirement, ensure_ascii=False, indent=2)}

MODEL SPEC:
{json.dumps(model, ensure_ascii=False, indent=2)}

AVAILABLE FIELDS:
{json.dumps(sorted(available_fields), ensure_ascii=False)}
'''
print(prompt)

In [ ]:
AI3_RESPONSE = r'''PASTE_AI_JSON_HERE'''

def validate_layouts(result):
    layouts = result.get('layouts')
    if not isinstance(layouts, list) or not 2 <= len(layouts) <= 3:
        raise ValueError('AI 3 phải trả về 2 hoặc 3 layout.')
    for layout in layouts:
        if not layout.get('pages'):
            raise ValueError(f'Layout {layout.get("id")} chưa có trang.')
        for page in layout['pages']:
            for visual in page.get('visuals', []):
                for field in visual.get('fields', []):
                    if field.get('field') not in available_fields:
                        raise ValueError(f'Field không có trong model: {field.get("field")}')
                position = visual.get('position', {})
                values = [position.get(k, 0) for k in ('x', 'y', 'width', 'height')]
                if any(not isinstance(v, (int, float)) or v < 0 or v > 1 for v in values):
                    raise ValueError(f'Vị trí phải nằm trong khoảng 0..1: {position}')
    return layouts

if AI3_RESPONSE.strip() == 'PASTE_AI_JSON_HERE':
    print('Dán JSON AI 3 vào AI3_RESPONSE rồi chạy lại cell này.')
else:
    layouts = validate_layouts(json.loads(AI3_RESPONSE))
    print('Các phương án:', [(item['id'], item['name']) for item in layouts])

In [ ]:
# Sau khi xác nhận AI3_RESPONSE hợp lệ, chọn ID layout được khách duyệt.
SELECTED_LAYOUT_ID = None  # Ví dụ: 'A'

if 'layouts' not in globals():
    raise RuntimeError('Chạy cell kiểm tra AI3_RESPONSE trước.')
selected = next((item for item in layouts if item['id'] == SELECTED_LAYOUT_ID), None)
if selected is None:
    raise ValueError('Đặt SELECTED_LAYOUT_ID trùng với ID phương án khách chọn.')

page_count = len(selected['pages'])
fig, axes = plt.subplots(page_count, 1, figsize=(12, 6 * page_count), squeeze=False)
for ax, page in zip(axes[:, 0], selected['pages']):
    ax.set_xlim(0, 1); ax.set_ylim(1, 0); ax.axis('off')
    ax.set_title(page['name'] + ' · ' + page.get('purpose', ''), loc='left', fontsize=14)
    for visual in page.get('visuals', []):
        pos = visual.get('position', {})
        x, y, width, height = (pos.get(k, 0) for k in ('x', 'y', 'width', 'height'))
        ax.add_patch(Rectangle((x, y), width, height, fill=False, linewidth=1.5))
        ax.text(x + 0.01, y + 0.025, f"{visual.get('type', 'visual')}: {visual.get('title', '')}", va='top', fontsize=9, wrap=True)
plt.tight_layout()
preview_dir = RUN_DIR / 'preview'
preview_dir.mkdir(exist_ok=True)
fig.savefig(preview_dir / f"layout_{SELECTED_LAYOUT_ID}.png", dpi=140, bbox_inches='tight')
plt.show()
(RUN_DIR / 'layout.json').write_text(json.dumps({'selected_layout_id': SELECTED_LAYOUT_ID, 'layout': selected}, ensure_ascii=False, indent=2), encoding='utf-8')
print('Đã lưu preview và layout.json trong', RUN_DIR)